# 2. Logistic Regression with TF-IDF
**Dataset:** CLINC150 — 151-class intent classification

**Approach:** TF-IDF features → Logistic Regression (multinomial, L2 regularized)

**Expected Accuracy:** ~80–90%

### Load Dataset

In [2]:
import os
import pandas as pd
from datasets import load_dataset

test_dataset  = load_dataset('parquet', data_files={'test': os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset('parquet', data_files={'train': os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
val_dataset   = load_dataset('parquet', data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

train_df = train_dataset['train'].to_pandas()
val_df   = val_dataset['validation'].to_pandas()
test_df  = test_dataset['test'].to_pandas()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
display(train_df.head(3))
display(val_df.head(3))
display(test_df.head(3))

Train: 10625 | Val: 3100 | Test: 5500


,text,intent
0,what are the steps for setting up direct depos...,108
1,how is a direct deposit set up,108
2,how would i go about setting up a direct deposit,108


,text,intent
0,"in spanish, meet me tomorrow is said how",61
1,"in french, how do i say, see you later",61
2,how do you say hello in japanese,61


,text,intent
0,how would you say fly in italian,61
1,what's the spanish word for pasta,61
2,how would they say butter in zambia,61


### Preprocessing

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def preprocess(sentences):
    base_stop_words = set(stopwords.words('english'))

    words_to_keep = {'what', 'where', 'who', 'why', 'how', 'when', 'do'}

    custom_stop_words = base_stop_words - words_to_keep

    allowed_symbols = {'?', '!'}

    lemmatizer = WordNetLemmatizer()
    result = []
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tags = nltk.pos_tag(tokens)
        cleaned = []
        for token, pos in pos_tags:
            t = token.lower()
            if (t not in custom_stop_words) and (t.isalpha() or t in allowed_symbols):
                if t.isalpha():
                    if pos.startswith('NN'):   lemma = lemmatizer.lemmatize(t, 'n')
                    elif pos.startswith('VB'): lemma = lemmatizer.lemmatize(t, 'v')
                    elif pos.startswith('JJ'): lemma = lemmatizer.lemmatize(t, 'a')
                    else:                      lemma = lemmatizer.lemmatize(t)
                else:
                    lemma = t
                cleaned.append(lemma)
        result.append(' '.join(cleaned))
    return result

train_texts = preprocess(train_df['text'].tolist())
val_texts   = preprocess(val_df['text'].tolist())
test_texts  = preprocess(test_df['text'].tolist())

y_train = train_df['intent'].tolist()
y_val   = val_df['intent'].tolist()
y_test  = test_df['intent'].tolist()

print('Sample:', train_texts[0])

Sample: what step set direct deposit paycheck


### Vectorization — TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True, analyzer='word')
char_vec = TfidfVectorizer(ngram_range=(3, 5), min_df=1, sublinear_tf=True, analyzer='char_wb')

X_train = sp.hstack([
    word_vec.fit_transform(train_texts),
    char_vec.fit_transform(train_texts)
])
X_val  = sp.hstack([word_vec.transform(val_texts),  char_vec.transform(val_texts)])
X_test = sp.hstack([word_vec.transform(test_texts), char_vec.transform(test_texts)])

print(f'Feature matrix shape: {X_train.shape}')

Feature matrix shape: (10625, 43367)


### Model Training — Logistic Regression
We use `saga` solver which supports L1/L2 penalties and scales well to many classes.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

print('Running GridSearch over C values...')
param_grid = {'C': [0.5, 1, 5, 10, 20]}
gs = GridSearchCV(
    LogisticRegression(
        solver='saga', penalty='l2', multi_class='multinomial',
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
gs.fit(X_train, y_train)
print(f'Best C: {gs.best_params_["C"]}  |  CV Accuracy: {gs.best_score_:.4f}')
lr_model = gs.best_estimator_

Running GridSearch over C values...
Fitting 3 folds for each of 5 candidates, totalling 15 fits


e:\Github Project\Intent-Router-Engine\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best C: 20  |  CV Accuracy: 0.8703


e:\Github Project\Intent-Router-Engine\.venv\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Evaluation

In [12]:
from sklearn.metrics import accuracy_score, classification_report

y_val_pred  = lr_model.predict(X_val)
y_test_pred = lr_model.predict(X_test)

print(f'Validation Accuracy : {accuracy_score(y_val, y_val_pred)*100:.2f}%')
print(f'Test Accuracy       : {accuracy_score(y_test, y_test_pred)*100:.2f}%')
print()
print('--- Validation Classification Report ---')
print(classification_report(y_val, y_val_pred))

Validation Accuracy : 84.26%
Test Accuracy       : 74.87%

--- Validation Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.55      0.69        20
           1       0.93      0.65      0.76        20
           2       0.85      0.85      0.85        20
           3       0.91      1.00      0.95        20
           4       0.87      0.65      0.74        20
           5       0.89      0.80      0.84        20
           6       0.94      0.75      0.83        20
           7       0.91      1.00      0.95        20
           8       0.95      0.90      0.92        20
           9       0.70      0.80      0.74        20
          10       0.83      1.00      0.91        20
          11       0.63      0.60      0.62        20
          12       0.86      0.95      0.90        20
          13       0.68      0.65      0.67        20
          14       1.00      1.00      1.00        20
          15       0.94      0.85  

### Top Predictive Features per Class
One advantage of LR is interpretability — we can see which tokens drive each prediction.

In [16]:
import numpy as np

feature_names = word_vec.get_feature_names_out().tolist() + char_vec.get_feature_names_out().tolist()
feature_names = np.array(feature_names)

# Show top 5 features for 3 sample classes
sample_classes = [0, 5, 82]  # restaurant_reviews, weather, greeting
class_labels   = ['restaurant_reviews', 'weather', 'greeting']

for cls_idx, cls_name in zip(sample_classes, class_labels):
    coef = lr_model.coef_[cls_idx]
    top5 = feature_names[np.argsort(coef)[-5:][::-1]]
    print(f'[{cls_name}] top features: {top5.tolist()}')

[restaurant_reviews] top features: ['review', 'rating', 'good', 'yakamoto', 'revie']
[weather] top features: ['weather', 'rain', 'snow', 'outside', 'condition']
[greeting] top features: ['how', 'good see', 'what', ' ho', 'well']


### Model Testing

In [17]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time', 
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations', 
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode', 
    15: 'what_are_your_hobbies', 16: 'order', 17: 'jump_start', 18: 'schedule_meeting', 19: 'meeting_schedule', 
    20: 'freeze_account', 21: 'what_song', 22: 'meaning_of_life', 23: 'restaurant_reservation', 24: 'traffic', 
    25: 'make_call', 26: 'text', 27: 'bill_balance', 28: 'improve_credit_score', 29: 'change_language', 
    30: 'no', 31: 'measurement_conversion', 32: 'timer', 33: 'flip_coin', 34: 'do_you_have_pets', 
    35: 'balance', 36: 'tell_joke', 37: 'last_maintenance', 38: 'exchange_rate', 39: 'uber', 
    40: 'car_rental', 41: 'credit_limit', 42: 'oos', 43: 'shopping_list', 44: 'expiration_date', 
    45: 'routing', 46: 'meal_suggestion', 47: 'tire_change', 48: 'todo_list', 49: 'card_declined', 
    50: 'rewards_balance', 51: 'change_accent', 52: 'vaccines', 53: 'reminder_update', 54: 'food_last', 
    55: 'change_ai_name', 56: 'bill_due', 57: 'who_do_you_work_for', 58: 'share_location', 59: 'international_visa', 
    60: 'calendar', 61: 'translate', 62: 'carry_on', 63: 'book_flight', 64: 'insurance_change', 
    65: 'todo_list_update', 66: 'timezone', 67: 'cancel_reservation', 68: 'transactions', 69: 'credit_score', 
    70: 'report_fraud', 71: 'spending_history', 72: 'directions', 73: 'spelling', 74: 'insurance', 
    75: 'what_is_your_name', 76: 'reminder', 77: 'where_are_you_from', 78: 'distance', 79: 'payday', 
    80: 'flight_status', 81: 'find_phone', 82: 'greeting', 83: 'alarm', 84: 'order_status', 
    85: 'confirm_reservation', 86: 'cook_time', 87: 'damaged_card', 88: 'reset_settings', 89: 'pin_change', 
    90: 'replacement_card_duration', 91: 'new_card', 92: 'roll_dice', 93: 'income', 94: 'taxes', 
    95: 'date', 96: 'who_made_you', 97: 'pto_request', 98: 'tire_pressure', 99: 'how_old_are_you', 
    100: 'rollover_401k', 101: 'pto_request_status', 102: 'how_busy', 103: 'application_status', 104: 'recipe', 
    105: 'calendar_update', 106: 'play_music', 107: 'yes', 108: 'direct_deposit', 109: 'credit_limit_change', 
    110: 'gas', 111: 'pay_bill', 112: 'ingredients_list', 113: 'lost_luggage', 114: 'goodbye', 
    115: 'what_can_i_ask_you', 116: 'book_hotel', 117: 'are_you_a_bot', 118: 'next_song', 119: 'change_speed', 
    120: 'plug_type', 121: 'maybe', 122: 'w2', 123: 'oil_change_when', 124: 'thank_you', 125: 'shopping_list_update', 
    126: 'pto_balance', 127: 'order_checks', 128: 'travel_alert', 129: 'fun_fact', 130: 'sync_device', 
    131: 'schedule_maintenance', 132: 'apr', 133: 'transfer', 134: 'ingredient_substitution', 135: 'calories', 
    136: 'current_location', 137: 'international_fees', 138: 'calculator', 139: 'definition', 140: 'next_holiday', 
    141: 'update_playlist', 142: 'mpg', 143: 'min_payment', 144: 'change_user_name', 145: 'restaurant_suggestion', 
    146: 'travel_notification', 147: 'cancel', 148: 'pto_used', 149: 'travel_suggestion', 150: 'change_volume'
}


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def predict(user_text: str):
    """Full pipeline: raw text → preprocess → vectorize → predict."""
    # 1. Preprocess (same pipeline used during training)
    cleaned = preprocess([user_text])                          # returns list of str

    # 2. Vectorize using the SAME fitted vectorizer
    X = sp.hstack([word_vec.transform(cleaned), char_vec.transform(cleaned)])

    # 3. Predict class & confidence
    prediction_id = lr_model.predict(X)[0]                    # scalar, not array
    probas         = lr_model.predict_proba(X)[0]             # shape: (num_classes,)
    confidence     = probas[prediction_id] * 100

    # 4. Top-3 alternatives
    top3_ids    = np.argsort(probas)[::-1][:3]
    top3        = [(intent_mapping.get(i, str(i)), probas[i] * 100) for i in top3_ids]

    return prediction_id, intent_mapping.get(prediction_id, 'Unknown Intent'), confidence, top3

# Create UI Elements
text_input = widgets.Text(
    value='',
    placeholder='Type your sentence here...',
    description='Sentence:',
    layout=widgets.Layout(width='80%')
)
button = widgets.Button(
    description='Predict Intent',
    button_style='success',
    tooltip='Click to predict the intent'
)
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        user_text = text_input.value.strip()
        if not user_text:
            print("⚠️  Please enter a sentence!")
            return

        pred_id, intent_label, confidence, top3 = predict(user_text)

        print(f"Input     : '{user_text}'")
        print(f"Prediction: [{intent_label.upper()}]  (id={pred_id})")
        print(f"Confidence: {confidence:.1f}%")
        print()
        print("Top 3 candidates:")
        for rank, (label, prob) in enumerate(top3, 1):
            bar = '█' * int(prob / 5)
            print(f"  {rank}. {label:<35s} {prob:5.1f}%  {bar}")

# Link the button to the function
button.on_click(on_button_clicked)

# Display the interface
display(widgets.HBox([text_input, button]))
display(output_area)

Output()